# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
# TODO
print("shape:", df.shape)

print("\ndtypes:")
print(df.dtypes)

print("\nnull count:")
print(df.isna().sum())

print("\nexact duplicate rows:", df.duplicated().sum())

shape: (8, 6)

dtypes:
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

null count:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

exact duplicate rows: 1


**What is wrong with this data?** List at least five specific problems:

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:

removed = df.duplicated().sum()

clean = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', removed)
# TODO: log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
# TODO: clean['price'] = ...


clean['price'] = (
    clean['price']
    .astype(str)
    .str.strip()
    .str.replace('$', '', regex=False)
    .astype(float)
)

assert clean['price'].dtype == float

log('price', 'stripped dollar signs/whitespace and converted to float', len(clean))
# assert clean['price'].dtype == float
# TODO: log(...) -- note that price arrived as text

[price] stripped dollar signs/whitespace and converted to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [6]:
# TODO: clean['qty'] = pd.to_numeric(...)

missing = None    # TODO: count of NaN quantities
negative = None   # TODO: count of negative quantities


clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()
negative = (clean['qty'] < 0).sum()

# Drop rows where quantity is missing
clean = clean[clean['qty'].notna()].copy()

log('quantity_missing', 'dropped rows with missing quantity', missing)
log('quantity_negative', 'kept negative quantity as a possible refund', negative)

# TODO: apply your decision, then log both separately

[quantity_missing] dropped rows with missing quantity (1 row(s))
[quantity_negative] kept negative quantity as a possible refund (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [7]:



print('before:', sorted(clean['category'].unique()))

# TODO: lowercase, strip, remove punctuation

clean['category'] = (
    clean['category']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]', '', regex=True)
)

CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Apparel',
    'raingear': 'RainGear'
}

clean['category'] = clean['category'].map(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))

log(
    'categories',
    'normalized case/punctuation and mapped category variants',
    len(clean)
)

# TODO: CATEGORY_MAP = {...} for the judgment calls

# print('after: ', sorted(clean['category'].unique()))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Apparel', 'Food', 'Merch', 'RainGear']
[categories] normalized case/punctuation and mapped category variants (6 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [8]:
# TODO
print('items before:', sorted(clean['item'].dropna().unique()))

clean['item'] = (
    clean['item']
    .astype('string')
    .str.strip()
    .str.lower()
)

ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'uva t-shirt': 'UVA T-Shirt',
    'rain poncho': 'Rain Poncho'
}

clean['item'] = clean['item'].map(ITEM_MAP)

# Drop the row with no item
missing_item = clean['item'].isna().sum()
clean = clean[clean['item'].notna()].copy()

print('items after:', sorted(clean['item'].unique()))

log('items', 'standardized item names and dropped missing item', missing_item)

items before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
items after: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt']
[items] standardized item names and dropped missing item (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [13]:
# TODO

clean['ts'] = pd.to_datetime(
    clean['ts'],
    format='mixed',
    errors='coerce'
)

failed = clean['ts'].isna().sum()

print('timestamps failed:', failed)

clean['hour'] = clean['ts'].dt.hour

log('timestamps', 'parsed timestamps and created hour column', failed)

timestamps failed: 1
[timestamps] parsed timestamps and created hour column (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [14]:
# TODO: assertions

clean['revenue'] = clean['qty'] * clean['price']

assert clean['price'].dtype == float
assert pd.api.types.is_numeric_dtype(clean['qty'])
assert pd.api.types.is_numeric_dtype(clean['revenue'])
assert clean['category'].notna().all()
assert clean['item'].notna().all()
assert clean['revenue'].notna().all()
assert len(clean) == 5



print(clean)

print('\nRevenue by row:')
print(clean[['order_id', 'item', 'qty', 'price', 'revenue']])

print('\nTotal revenue:', clean['revenue'].sum())

# TODO: clean['revenue'] = ...
# TODO: print rows, units, revenue, distinct categories

   order_id          item  category  qty  price                  ts  revenue  \
0         1  Cheeseburger      Food  2.0    7.5 2026-09-05 12:03:00     15.0   
2         2  Cheeseburger      Food  1.0    7.5 2026-09-05 12:40:00      7.5   
4         4   UVA T-Shirt   Apparel  2.0   24.0 2026-09-05 13:05:00     48.0   
5         5   Rain Poncho  RainGear -3.0    6.0 2026-09-05 13:20:00    -18.0   
6         6   Rain Poncho  RainGear  4.0    6.0                 NaT     24.0   

   hour  
0  12.0  
2  12.0  
4  13.0  
5  13.0  
6   NaN  

Revenue by row:
   order_id          item  qty  price  revenue
0         1  Cheeseburger  2.0    7.5     15.0
2         2  Cheeseburger  1.0    7.5      7.5
4         4   UVA T-Shirt  2.0   24.0     48.0
5         5   Rain Poncho -3.0    6.0    -18.0
6         6   Rain Poncho  4.0    6.0     24.0

Total revenue: 76.5


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [15]:
print(show_log())

                step                                           decision  rows
0         duplicates                       dropped exact duplicate rows     1
1              price  stripped dollar signs/whitespace and converted...     7
2   quantity_missing                 dropped rows with missing quantity     1
3  quantity_negative        kept negative quantity as a possible refund     1
4         categories  normalized case/punctuation and mapped categor...     6
5              items   standardized item names and dropped missing item     1
6         timestamps          parsed timestamps and created hour column     1


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [18]:
# Checkpoint


rows_after = len(clean)
revenue_after = clean['revenue'].sum()
biggest_decision = 'kept the negative quantity as a refund'
revenue_other_way = revenue_after - (-3 * 6)

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)




rows after cleaning: 5
revenue: 76.5
decision that mattered: kept the negative quantity as a refund
revenue the other way: 94.5


After cleaning the data, there were 5 rows remaining with a total revenue of 76.50. The decision that affected the revenue the most was keeping the negative quantity as a refund; keeping it resulted in 76.50
 in revenue, while dropping the refund would have resulted in 94.50.
